In [1]:
import ccxt

exchange = ccxt.binance()

# L2 order book snapshot — 20 levels deep
ob = exchange.fetch_order_book('BTC/USDT', limit=20)
# ob = {
#   'bids': [[price, size], ...],  # sorted high→low
#   'asks': [[price, size], ...],  # sorted low→high
#   'timestamp': 1712345678000,
#   'nonce': ...
# }

# Recent trades (for buy/sell imbalance)
trades = exchange.fetch_trades('BTC/USDT', limit=500)
# each trade has: price, amount, side ('buy'/'sell'), timestamp

# OHLCV for price context
ohlcv = exchange.fetch_ohlcv('BTC/USDT', timeframe='1m', limit=100)

In [27]:
from pydantic import BaseModel
from typing import Literal
import time

class TradeRecord(BaseModel):
    symbol:    str
    timestamp: int
    price:     float
    amount:    float
    side:      Literal['buy', 'sell']
    trade_id:  str

class OrderBookSnapshot(BaseModel):
    symbol:    str
    timestamp: int
    bids:      list[list[float]]  # [[price, size], ...]
    asks:      list[list[float]]

class OHLCVCandle(BaseModel):
    symbol:    str = 'BTC/USDT'
    timestamp: int
    open:      float
    high:      float
    low:       float
    close:     float
    volume:    float
    timeframe: str = '1m'

In [25]:
import asyncio, json, websockets
from kafka import KafkaProducer

# producer = KafkaProducer(
#     bootstrap_servers='localhost:9092',
#     value_serializer=lambda v: json.dumps(v).encode()
# )

async def stream_order_book(symbol: str = 'btcusdt'):
    # @depth20@100ms = 20-level book, pushed every 100ms
    url = f"wss://stream.binance.com:9443/ws/{symbol}@depth20@100ms"

    async with websockets.connect(url, ping_interval=20) as ws:
        async for raw in ws:
            msg = json.loads(raw)
            print(msg)
            record = {
                'symbol':    symbol.upper(),
                'timestamp': int(time.time() * 1000),           # event time ms
                'bids':      msg['bids'],                       # [[price, qty], ...]
                'asks':      msg['asks'],
                'source':    'binance_ws'
            }
            print(record)
            break
            # producer.send('raw_order_book', value=record)

async def stream_trades(symbol: str = 'btcusdt'):
    url = f"wss://stream.binance.com:9443/ws/{symbol}@trade"

    async with websockets.connect(url, ping_interval=20) as ws:
        async for raw in ws:
            msg = json.loads(raw)
            record = {
                'symbol':    symbol.upper(),
                'timestamp': msg['T'],
                'price':     float(msg['p']),
                'qty':       float(msg['q']),
                'is_buy':    not msg['m'],   # m=True means maker was seller → taker bought
                'source':    'binance_ws'
            }
            producer.send('raw_prices', value=record)

async def main():
    await asyncio.gather(
        stream_order_book('btcusdt'),
        stream_trades('btcusdt'),
    )

# asyncio.run(main())

In [28]:
await stream_order_book('btcusdt')

{'lastUpdateId': 91489925135, 'bids': [['69753.78000000', '5.03848000'], ['69753.77000000', '0.00008000'], ['69753.73000000', '0.00016000'], ['69753.72000000', '0.03399000'], ['69753.71000000', '0.10008000'], ['69753.25000000', '0.00300000'], ['69752.63000000', '0.00609000'], ['69752.56000000', '0.00008000'], ['69751.50000000', '0.00300000'], ['69751.38000000', '0.00008000'], ['69751.18000000', '0.00008000'], ['69750.59000000', '0.00016000'], ['69750.58000000', '0.02045000'], ['69749.95000000', '0.00030000'], ['69749.87000000', '0.00672000'], ['69749.85000000', '0.01448000'], ['69749.80000000', '0.00016000'], ['69749.75000000', '0.00300000'], ['69749.59000000', '0.05397000'], ['69749.23000000', '0.00024000']], 'asks': [['69753.79000000', '1.11028000'], ['69753.80000000', '0.00056000'], ['69753.96000000', '0.00010000'], ['69754.00000000', '0.00120000'], ['69754.32000000', '0.00010000'], ['69754.71000000', '0.00350000'], ['69754.93000000', '0.00008000'], ['69754.94000000', '0.00008000'],